# 02. 전처리

음식 데이터 정제와 전처리 방법을 실험한다.

- 결측치·중복 처리 방식 실험
- 음식명·설명 텍스트 정규화
- 추천에 사용할 컬럼 선별 및 정리
- 검증된 로직은 `src/preprocessing/`으로 분리

입력: `data/raw/` / 출력: `data/processed/`

01 분석 결과를 바탕으로 다음 순서로 진행한다.

1. 메뉴 그룹 분류: 대분류 기준 1차 분류 후 조리법 대분류 안의 반찬을 대표식품명으로 보완
2. 식품명 정제: 원본 보존, 추천용 메뉴명과 온도, 사이즈 속성 분리
3. 식품중량 분리
4. 중복 처리: 업체, 출처, 영양성분을 고려한 통합 기준
5. 컬럼 선별과 `해당없음` 처리
6. `data/processed/` 저장, 무결성 검증, 전후 비교
7. 식사 후보 분포 확인과 라벨링 샘플 추출 방안

각 단계의 검증된 로직은 `src/preprocessing/food_data.py`에 있으며 이 노트북은 그 함수를 실제 데이터에 적용하며 근거를 기록한다.

## 기본 설정

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 저장소 루트 또는 노트북 폴더에서 실행 가능
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Menu_recommend 저장소 안에서 실행해 주세요")
DATA_DIR = PROJECT_ROOT / "data"
RAW_PATH = DATA_DIR / "raw" / "food_nutrition.csv"
PROCESSED_DIR = DATA_DIR / "processed"
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import food_data as fd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

In [2]:
raw = fd.load_raw(RAW_PATH)
raw.shape

(19617, 50)

## 1. 메뉴 그룹 분류

01 분석에서 빵 및 과자류와 음료 및 차류가 전체의 약 73%였다. 1차 추천 대상을 식사 메뉴로 잡기 위해 `식품대분류명`을 기준으로 다섯 그룹으로 나눈다.

| 그룹 | 기준 |
|---|---|
| 식사 | 밥, 면·만두, 국·탕, 찌개·전골, 죽·스프, 볶음, 구이, 튀김, 찜, 조림, 전·적·부침 |
| 반찬 | 생채·무침, 나물·숙채, 김치, 장아찌·절임, 젓갈, 장류·양념 |
| 디저트 | 빵 및 과자류, 유제품류 및 빙과류 |
| 음료 | 음료 및 차류 |
| 기타 | 수·조·어·육류, 곡류·서류 제품, 채소·해조류, 과일류, 두류·견과 (원재료 성격, 17행) |

빵 및 과자류 안에는 피자, 버거, 샌드위치처럼 한 끼 식사로 먹는 메뉴가 섞여 있다. 어떤 대표식품명이 있는지 확인한다.

In [3]:
raw.loc[raw["식품대분류명"] == "빵 및 과자류", "대표식품명"].value_counts().head(20)

대표식품명
피자            4710
케이크            657
버거             295
도넛             285
샌드위치           233
와플             199
마카롱            187
크로플            162
베이글            136
햄버거            133
크림빵            113
비스킷/쿠키/크래커     106
식빵             105
크로와상            96
페이스트리           79
치즈빵             66
크로켓(고로케)        62
핫도그             61
머핀              60
프레즐             57
Name: count, dtype: int64

피자, 버거, 햄버거, 샌드위치, 핫도그, 토스트는 식사로 분류한다. 케이크, 도넛, 와플 등은 디저트로 둔다.

반찬은 단독 메뉴로 추천하기 어렵고 급식 데이터 성격이 강해 1차 대상에서 제외하되, 그룹 라벨을 남겨 나중에 활용할 수 있게 한다.

In [4]:
work = raw.copy()
work["프랜차이즈여부"] = work["업체명"] != fd.NOT_APPLICABLE

# 대분류만으로 나눈 1차 분류 (보완 전 기준, 비교용)
category = work["식품대분류명"]
category_group = pd.Series(fd.MENU_GROUP_OTHER, index=work.index, dtype="object")
category_group[category.isin(fd.MEAL_CATEGORIES)] = fd.MENU_GROUP_MEAL
category_group[category.isin(fd.SIDE_DISH_CATEGORIES)] = fd.MENU_GROUP_SIDE
category_group[category.isin(fd.DESSERT_CATEGORIES)] = fd.MENU_GROUP_DESSERT
category_group[category.isin(fd.BEVERAGE_CATEGORIES)] = fd.MENU_GROUP_BEVERAGE
category_group[(category == "빵 및 과자류") & work["대표식품명"].isin(fd.MEAL_BREAD_REPRESENTATIVES)] = fd.MENU_GROUP_MEAL
work["메뉴그룹_대분류기준"] = category_group

work["메뉴그룹_대분류기준"].value_counts()

메뉴그룹_대분류기준
식사     9117
음료     5776
디저트    3840
반찬      867
기타       17
Name: count, dtype: int64

### 1-2. 조리법 대분류 안의 반찬 보완

볶음류, 조림류, 구이류, 튀김류, 찜류, 전·적 및 부침류는 조리법 기준 분류라 잔멸치볶음, 두부조림, 달걀말이 같은 반찬이 제육볶음, 고등어조림, 파전 같은 식사와 섞여 있다.

이 여섯 대분류에서는 `대표식품명`을 보고 반찬을 다시 골라낸다. `메뉴명` 대신 `대표식품명`을 쓰는 이유는 표기가 정규화되어 있고 고유값이 441개라 전부 눈으로 검토할 수 있기 때문이다.

판단 규칙은 세 단계다.

1. 강한 반찬 키워드(장조림, 맛탕, 뱅어포, 콘치즈, 떡강정)가 있으면 반찬. `소고기 장조림`처럼 식사 키워드가 같이 있어도 반찬이다.
2. 반찬 키워드(멸치, 감자, 두부, 달걀, 어묵, 소시지, 버섯 등)가 있고 식사 키워드가 없으면 반찬.
3. 그 외는 식사. `감자그라탕`, `마파두부`, `두부 탕수`처럼 반찬 재료가 이름에 있어도 식사 키워드(그라탕, 마파두부, 탕수)가 있으면 식사로 유지한다.

키워드는 여섯 대분류의 대표식품명 전체를 검토해 정했고 `src/preprocessing/food_data.py`의 상수로 관리한다. 규칙은 여섯 대분류에만 적용하므로 `김치 볶음밥`(밥류), `달걀국`(국 및 탕류)은 영향을 받지 않는다.

In [5]:
work["메뉴그룹"] = fd.assign_menu_group(work)

cooking = work[work["식품대분류명"].isin(fd.COOKING_CATEGORIES)]
side_change = pd.DataFrame({
    "보완 전 식사": cooking.groupby("식품대분류명").size(),
    "보완 후 식사": cooking[cooking["메뉴그룹"] == fd.MENU_GROUP_MEAL].groupby("식품대분류명").size(),
    "반찬 전환": cooking[cooking["메뉴그룹"] == fd.MENU_GROUP_SIDE].groupby("식품대분류명").size(),
}).fillna(0).astype(int)
side_change.loc["합계"] = side_change.sum()
side_change

,보완 전 식사,보완 후 식사,반찬 전환
식품대분류명,,,
구이류,289,242,47
볶음류,506,252,254
전·적 및 부침류,198,69,129
조림류,229,79,150
찜류,177,125,52
튀김류,526,391,135
합계,1925,1158,767


In [6]:
# 반찬으로 전환된 대표식품명 (대분류별)
for cat in fd.COOKING_CATEGORIES:
    names = sorted(cooking.loc[(cooking["식품대분류명"] == cat) & (cooking["메뉴그룹"] == fd.MENU_GROUP_SIDE), "대표식품명"].unique())
    print(f"[{cat}] {len(names)}개: {', '.join(names)}")
    print()

[볶음류] 46개: 가지볶음, 감자볶음, 건새우볶음, 고추장볶음, 김치볶음, 껍질콩볶음, 꽈리고추볶음, 느타리버섯볶음, 닭모래집볶음, 닭발볶음, 당근볶음, 두부볶음, 마늘쫑볶음, 매운 어묵볶음, 머위나물볶음, 멸치볶음, 멸치볶음(멸치만), 모듬버섯볶음, 미역줄기볶음, 버섯볶음, 베이컨채소볶음, 브로콜리 버섯볶음, 브로콜리볶음, 새송이버섯볶음, 소간 채소볶음, 소시지 케첩볶음, 소시지볶음, 애호박 볶음, 애호박볶음, 양념두부, 양송이버섯볶음, 양파볶음, 어묵 감자볶음, 어묵볶음, 오징어채볶음, 우엉볶음, 잔멸치볶음, 죽순볶음, 파래볶음, 표고버섯볶음, 풋고추볶음, 피망볶음, 햄볶음, 햄채소볶음, 호박고지볶음, 호박볶음

[조림류] 40개: 감자조림, 게조림, 고구마조림, 고추조림, 곤약조림, 꽈리고추 오징어조림, 꽈리고추조림, 다시마조림, 단호박조림, 달걀조림, 돼지고기 장조림, 돼지등심 장조림, 두부조림, 땅콩조림, 마늘쫑조림, 메추리알 돼지고기장조림, 메추리알 어묵조림, 메추리알장조림, 멸치조림, 모듬 콩조림, 무 어묵조림, 무조림, 미트볼조림, 버섯조림, 삶은 땅콩 조림, 새우조림, 소고기 장조림, 소시지조림, 양미리조림, 어묵조림, 연근조림, 오징어채조림, 오징어포조림, 완자조림, 우엉조림, 유부조림, 장조림, 쥐포조림, 콩조림, 콩조림(콩자반)

[구이류] 17개: 감자구이, 김구이, 느타리버섯 구이, 닭발구이, 더덕구이, 런천미트구이, 뱅어포구이, 버섯구이, 버섯구이표고버섯, 베이컨떡말이구이, 새송이버섯 구이, 쑥전(부침개), 옥수수구이, 우엉 양념구이, 조미 김구이, 콘치즈구이, 햄구이

[튀김류] 34개: 감자연근튀김, 감자튀김, 게맛살튀김, 고구마깻잎튀김, 고구마맛탕, 고구마튀김, 고추튀김, 김말이튀김, 김부각, 김튀김, 깻잎튀김, 느타리버섯튀김, 다시마튀각, 달걀튀김, 닭껍데기튀김, 닭모래집튀김, 닭발튀김, 도라지튀김, 두부튀김, 떡강정, 미역튀각, 뱅어포튀김, 소시지튀김, 식빵튀김, 쑥튀김, 양파링튀김, 어묵튀김, 연근튀김, 우유튀김, 잔멸치

In [7]:
# 반찬 키워드가 이름에 있지만 식사 키워드로 식사에 남은 대표식품명
has_side_kw = cooking["대표식품명"].map(lambda n: any(k in n for k in fd.SIDE_DISH_KEYWORDS))
kept_meal = cooking[has_side_kw & (cooking["메뉴그룹"] == fd.MENU_GROUP_MEAL)]
sorted(kept_meal["대표식품명"].unique())

['가오리찜',
 '가오리콩나물찜',
 '가지탕수',
 '감자그라탕',
 '꽃게콩나물찜',
 '돼지고기 피망잡채',
 '두부 탕수',
 '두부김치',
 '마파두부',
 '버섯 잡채',
 '소고기 완자전',
 '소고기산적',
 '소고기채소볶음',
 '양파 소고기전',
 '어묵잡채',
 '채소 꼬치구이',
 '콩나물잡채',
 '해물 완자전',
 '해물 채소전',
 '해물콩나물찜']

In [8]:
# 전환된 행의 실제 메뉴명 예시
changed = cooking[cooking["메뉴그룹"] == fd.MENU_GROUP_SIDE]
changed[["식품명", "대표식품명", "식품대분류명", "프랜차이즈여부"]].sample(12, random_state=0)

,식품명,대표식품명,식품대분류명,프랜차이즈여부
19321,고구마조림,고구마조림,조림류,False
9519,잔멸치볶음_꽈리고추,잔멸치볶음,볶음류,False
9510,잔멸치볶음_풋고추,잔멸치볶음,볶음류,False
405,햄전,햄전,전·적 및 부침류,False
18144,달걀찜,달걀찜,찜류,False
9425,장조림_돼지고기_메추리알,장조림,조림류,False
14628,뱅어포구이,뱅어포구이,구이류,False
18381,꼬막찜,꼬막찜,찜류,False
15413,멸치볶음,멸치볶음,볶음류,False
10609,어묵조림,어묵조림,조림류,False


프랜차이즈 사이드 메뉴(감자튀김, 치즈스틱, 치즈볼)도 이 규칙으로 반찬 그룹에 들어간다. 반찬 그룹은 `단독 식사가 아닌 곁들임·간식` 의미로 사용한다.

In [9]:
changed.loc[changed["프랜차이즈여부"], "대표식품명"].value_counts()

대표식품명
감자튀김      21
치즈볼       10
치즈스틱       9
닭발구이       5
떡강정        4
닭모래집튀김     3
쥐포튀김       1
우유튀김       1
미트볼조림      1
닭발튀김       1
닭껍데기튀김     1
Name: count, dtype: int64

In [10]:
group_counts = (
    work.groupby(["메뉴그룹", "식품대분류명"]).size().rename("행 수").reset_index()
    .sort_values(["메뉴그룹", "행 수"], ascending=[True, False])
)
group_counts

,메뉴그룹,식품대분류명,행 수
3,기타,수·조·어·육류,8
0,기타,"곡류, 서류 제품",6
1,기타,과일류,1
2,기타,"두류, 견과 및 종실류",1
4,기타,"채소, 해조류",1
5,디저트,빵 및 과자류,3177
6,디저트,유제품류 및 빙과류,663
11,반찬,생채·무침류,501
10,반찬,볶음류,254
9,반찬,나물·숙채류,248


In [11]:
pd.DataFrame({
    "보완 전": work["메뉴그룹_대분류기준"].value_counts(),
    "보완 후": work["메뉴그룹"].value_counts(),
}).fillna(0).astype(int)

,보완 전,보완 후
식사,9117,8350
음료,5776,5776
디저트,3840,3840
반찬,867,1634
기타,17,17


## 2. 식품명 정제

원본 `식품명`은 그대로 두고 추천용 `메뉴명`을 추가한다. 01 분석에서 확인한 표기 형태에 따라 규칙을 다르게 적용한다.

- 프랜차이즈 (`업체명`이 있는 행): `카테고리_메뉴명 온도 (사이즈)` 형태. 첫 `_` 앞은 `이름접두어`로 분리
- 비프랜차이즈: `_`는 재료 변형 구분자 (예: `된장국_근대`). 공백으로 치환
- `핫(HOT)`, `아이스(ICED)`는 `온도` 컬럼으로 분리
- 괄호 안 값이 사이즈 표기면 `사이즈` 컬럼으로 분리. 그 외 괄호 (조각, 8개입 등)는 이름에 유지

먼저 괄호 안에 어떤 값이 있는지 확인해 사이즈 표기를 결정한다.

In [12]:
paren_tokens = raw["식품명"].str.findall(r"\(([^()]*)\)").explode().dropna()
paren_tokens.value_counts().head(40)

식품명
L             3049
ICED          1890
R             1485
HOT           1449
M              590
F              227
Tall           179
P              153
XL             145
Venti          142
ML             133
Grande         128
EX             103
Mini Venti      94
J               91
S               68
조각              62
고로케             62
더벤티             52
1인              32
V               31
Max             29
G               28
코끼리             28
치즈              26
소               23
액상              23
H               18
대               16
8개입             15
초대용량            15
5개입             14
홀               14
닭갈비             14
3개입             13
20개입            10
2개입             10
싱글              10
더블              10
1개입              8
Name: count, dtype: int64

L, R, M, S, XL, Tall, Grande, Venti 등 음료·피자 사이즈 표기만 사이즈로 인정한다. `조각`, `N개입`, `1인`처럼 수량 정보는 이름에 남긴다.

`아이스`, `핫` 단독 표기는 `아이스크림`과 겹치므로 `(HOT)`, `(ICED)` 토큰이 있는 경우만 온도로 추출한다.

In [13]:
print("사이즈로 인정하는 표기:", sorted(fd.SIZE_TOKENS))

사이즈로 인정하는 표기: ['EX', 'F', 'G', 'Grande', 'H', 'J', 'L', 'M', 'ML', 'Max', 'Mini Venti', 'P', 'R', 'S', 'Short', 'Solo', 'Tall', 'V', 'Venti', 'XL', '대', '더벤티', '소', '중', '초대용량']


In [14]:
work = fd.add_name_columns(work)

work.loc[work["프랜차이즈여부"], ["식품명", "메뉴명", "이름접두어", "온도", "사이즈"]].sample(12, random_state=0)

,식품명,메뉴명,이름접두어,온도,사이즈
3994,피자_버팔로 피자 치즈크러스트 (R),버팔로 피자 치즈크러스트,피자,NaN,R
1854,피자_크리스피 치즈 페퍼로니,크리스피 치즈 페퍼로니,피자,NaN,NaN
7990,커피_아메리카노 핫(HOT),아메리카노,커피,HOT,NaN
10914,아이스크림_하루한번하늘 그릭요거트,하루한번하늘 그릭요거트,아이스크림,NaN,NaN
1418,피자_페페로니 피자 씬도우 (L),페페로니 피자 씬도우,피자,NaN,L
2537,피자_전주불백피자 (XL),전주불백피자,피자,NaN,XL
14859,밀크티/버블티_타로 버블티,타로 버블티,밀크티/버블티,NaN,NaN
15846,레몬차_레몬티 핫(HOT),레몬티,레몬차,HOT,NaN
6633,케이크_오리지널 티라미수 케이크,오리지널 티라미수 케이크,케이크,NaN,NaN
6039,크로플_마약크림 딸기 크로플,마약크림 딸기 크로플,크로플,NaN,NaN


In [15]:
work.loc[~work["프랜차이즈여부"], ["식품명", "메뉴명", "이름접두어", "온도", "사이즈"]].sample(8, random_state=0)

,식품명,메뉴명,이름접두어,온도,사이즈
9170,청포묵 무침,청포묵 무침,NaN,NaN,NaN
9122,취나물무침,취나물무침,NaN,NaN,NaN
15563,막국수,막국수,NaN,NaN,NaN
9334,죽순볶음,죽순볶음,NaN,NaN,NaN
13559,삼계탕,삼계탕,NaN,NaN,NaN
12929,소시지볶음,소시지볶음,NaN,NaN,NaN
18194,다시마튀각,다시마튀각,NaN,NaN,NaN
18178,달걀 샐러드,달걀 샐러드,NaN,NaN,NaN


In [16]:
print("온도 분포"); print(work["온도"].value_counts(dropna=False))
print()
print("사이즈 분포 (상위)"); print(work["사이즈"].value_counts(dropna=False).head(12))

온도 분포
온도
NaN     16278
ICED     1890
HOT      1449
Name: count, dtype: int64

사이즈 분포 (상위)
사이즈
NaN       12806
L          3049
R          1485
M           590
F           227
Tall        179
P           153
XL          145
Venti       142
ML          133
Grande      128
EX          103
Name: count, dtype: int64


In [17]:
# 정제 후 메뉴명에 남은 괄호 표기 확인 (사이즈로 처리되지 않은 값)
work["메뉴명"].str.findall(r"\(([^()]*)\)").explode().dropna().value_counts().head(15)

메뉴명
조각      62
1인      32
코끼리     28
치즈      26
8개입     15
5개입     14
홀       14
3개입     13
닭갈비     11
20개입    10
2개입     10
싱글      10
더블      10
1개입      8
6개입      7
Name: count, dtype: int64

In [18]:
# 정제 후 메뉴명 고유값 변화
print(f"식품명 고유값: {work['식품명'].nunique():,}")
print(f"메뉴명 고유값: {work['메뉴명'].nunique():,}")
print(f"메뉴명 + 업체명 고유값: {work[['메뉴명', '업체명']].drop_duplicates().shape[0]:,}")

식품명 고유값: 15,647
메뉴명 고유값: 11,454
메뉴명 + 업체명 고유값: 13,096


## 3. 식품중량 분리

`식품중량`은 `291.90ml`처럼 숫자와 단위가 붙은 문자열이다. 숫자 `중량값`과 단위 `중량단위`(g, ml)로 분리만 하고 환산이나 1인분 해석은 하지 않는다. `1인(회)분량 참고량`이 전부 결측이라 `식품중량`이 1인분인지 확인할 근거가 없다.

In [19]:
work = pd.concat([work, fd.parse_weight(work["식품중량"])], axis=1)

parse_failed = work["식품중량"].notna() & work["중량값"].isna()
print(f"원본 결측: {work['식품중량'].isna().sum()}")
print(f"형식 불일치로 분리 실패: {parse_failed.sum()}")
print()
print(work["중량단위"].value_counts(dropna=False))

원본 결측: 52
형식 불일치로 분리 실패: 0

중량단위
g      13825
ml      5740
NaN       52
Name: count, dtype: int64


In [20]:
work[["식품명", "영양성분함량기준량", "식품중량", "중량값", "중량단위"]].sample(6, random_state=1)

,식품명,영양성분함량기준량,식품중량,중량값,중량단위
4765,피자_닭발 피자 (L),100g,932g,932.0,g
5794,타르트_스모어마시멜로우타르트,100g,50g,50.0,g
4641,피자_도이치 피자 밀도우 (L),100g,1120g,1120.0,g
942,피자_할루미체다크림크러스트(L),100g,1020g,1020.0,g
12068,스무디_제주 그린티 스무디,100g,473g,473.0,g
2320,피자_치왕체다크림크러스트,100g,1305g,1305.0,g


In [21]:
# 기준량 단위와 중량 단위가 다른 경우 확인
pd.crosstab(work["영양성분함량기준량"], work["중량단위"].fillna("결측"))

중량단위,g,ml,결측
영양성분함량기준량,,,
100g,13825,0,52
100ml,0,5740,0


## 4. 중복 처리 기준

01 분석에서 식품명이 같은 행이 3,970행 있었지만 원인이 두 가지였다.

- 급식 데이터: 같은 음식이 초등·중고등·산업체 급식으로 반복되며 영양성분도 동일
- 프랜차이즈: 같은 메뉴명이 여러 업체에 있으며 영양성분은 다름

음식명만으로 제거하면 업체별로 다른 메뉴가 사라진다. 따라서 다음 컬럼이 모두 같은 행만 하나로 통합한다.

- `식품명`, `업체명`, `영양성분함량기준량`, `식품중량`
- 영양성분 24개 컬럼 전체 (결측은 결측끼리 같은 것으로 간주)

통합 시 `식품코드` 오름차순 첫 행을 남기고, 제거된 행은 `대표식품코드`에 매핑해 별도 파일로 저장한다.

In [22]:
before_dedup = len(work)
work, dedup_map = fd.deduplicate(work)

print(f"통합 전: {before_dedup:,}행")
print(f"통합 후: {len(work):,}행")
print(f"제거 (매핑표 기록): {len(dedup_map):,}행")
print(f"중복 판단에 사용한 영양성분 컬럼: {len(fd.NUTRITION_COLUMNS_ALL)}개")

통합 전: 19,617행
통합 후: 18,998행
제거 (매핑표 기록): 619행
중복 판단에 사용한 영양성분 컬럼: 24개


In [23]:
dedup_map.head(10)

,식품코드,대표식품코드,식품명,식품기원명,업체명
0,D501-003000000-0001,D401-003000000-0001,곤드레밥,초등학교급식(재료량 기반 산출 함량),해당없음
1,D501-007480000-0001,D401-007480000-0001,김밥_채소,초등학교급식(재료량 기반 산출 함량),해당없음
2,D501-017030000-0001,D401-017030000-0001,볶음밥_계란,초등학교급식(재료량 기반 산출 함량),해당없음
3,D501-017480000-0001,D401-017480000-0001,볶음밥_채소,초등학교급식(재료량 기반 산출 함량),해당없음
4,D501-032600000-0001,D401-032600000-0001,잡곡밥_보리,초등학교급식(재료량 기반 산출 함량),해당없음
5,D501-035000000-0001,D401-035000000-0001,주먹밥,초등학교급식(재료량 기반 산출 함량),해당없음
6,D501-068000000-0001,D401-068000000-0001,버섯 덮밥,초등학교급식(재료량 기반 산출 함량),해당없음
7,D501-073000000-0001,D401-073000000-0001,양송이 덮밥,초등학교급식(재료량 기반 산출 함량),해당없음
8,D501-077000000-0001,D401-077000000-0001,완두콩밥,초등학교급식(재료량 기반 산출 함량),해당없음
9,D502-077000000-0001,D402-077000000-0001,계란빵,초등학교급식(재료량 기반 산출 함량),해당없음


In [24]:
# 제거된 행의 출처: 급식 반복 데이터만 해당하는지 확인
dedup_map["식품기원명"].value_counts()

식품기원명
산업체급식(재료량 기반 산출 함량)     295
중고등학교급식(재료량 기반 산출함량)    234
초등학교급식(재료량 기반 산출 함량)     90
Name: count, dtype: int64

In [25]:
# 통합 예시: 흰죽
sample_code = dedup_map.loc[dedup_map["식품명"] == "흰죽", "대표식품코드"].iloc[0]
print("대표 행")
display(work.loc[work["식품코드"] == sample_code, ["식품코드", "식품명", "식품기원명", "에너지(kcal)", "나트륨(mg)"]])
print("매핑된 행")
display(dedup_map[dedup_map["대표식품코드"] == sample_code])

대표 행


,식품코드,식품명,식품기원명,에너지(kcal),나트륨(mg)
0,D504-212000000-0001,흰죽,초등학교급식(재료량 기반 산출 함량),64,130.0


매핑된 행


,식품코드,대표식품코드,식품명,식품기원명,업체명
134,D604-212000000-0001,D504-212000000-0001,흰죽,중고등학교급식(재료량 기반 산출함량),해당없음
362,D704-212000000-0001,D504-212000000-0001,흰죽,산업체급식(재료량 기반 산출 함량),해당없음


In [26]:
# 식품명만 같고 통합되지 않은 예시: 업체별 영양성분이 다름
work.loc[work["식품명"] == "커피_아메리카노 핫(HOT)", ["식품코드", "업체명", "식품중량", "에너지(kcal)", "당류(g)", "나트륨(mg)"]].head(8)

,식품코드,업체명,식품중량,에너지(kcal),당류(g),나트륨(mg)
7982,D220-748080000-0003,컴포즈커피,591ml,3,0.0,1.0
7983,D220-748080000-0069,할리스,354ml,3,0.0,1.0
7984,D220-748080000-0076,더벤티,600ml,2,NaN,NaN
7985,D220-748000000-0945,베러댄와플,340g,3,0.0,1.0
7986,D220-748000000-0943,매머드 익스프레스,473g,2,NaN,0.0
7987,D220-748000000-0941,달콤,355g,2,0.0,0.0
7988,D220-748080000-0079,커피베이,360ml,1,0.0,0.0
7989,D220-748080000-0080,빽다방,473ml,3,0.0,1.0


## 5. 컬럼 선별과 `해당없음` 처리

01 분석에서 정리한 역할에 따라 추천에 필요한 컬럼만 남긴다.

| 역할 | 유지 컬럼 |
|---|---|
| 식별자 | `식품코드` |
| 음식명 | `식품명`(원본), `메뉴명`, `이름접두어`, `온도`, `사이즈` |
| 분류 | `메뉴그룹`, `식품대분류명`, `대표식품명`, `식품중분류명` |
| 출처 | `식품기원명`, `업체명`, `프랜차이즈여부`, `출처명`, `데이터생성방법명` |
| 제공량 | `영양성분함량기준량`, `중량값`, `중량단위` |
| 영양성분 | 원본 24개 중 9개: 에너지, 단백질, 지방, 탄수화물, 당류, 식이섬유, 나트륨, 콜레스테롤, 포화지방산 |

제외: 전체 결측인 `1인(회)분량 참고량`, 고유값 1개인 컬럼, 명칭과 1:1 대응하는 코드 컬럼, 관리용 날짜, `해당없음` 비율이 높은 소분류·세분류, 결측 68% 이상인 미량영양소.

영양성분 결측은 그대로 둔다. 프랜차이즈 데이터의 미량영양소 결측은 값이 0이 아니라 측정하지 않은 것이다.

`해당없음`은 컬럼 의미에 따라 다르게 처리한다.

- `업체명`: 업체가 없다는 뜻이므로 결측(NaN)으로 바꾸고 `프랜차이즈여부`로 구분
- `식품중분류명`: 분류가 없다는 뜻이므로 결측으로 바꿈
- `식품소분류명`, `식품세분류명`: 대부분 `해당없음`이라 컬럼 자체를 제외

In [27]:
work = fd.replace_not_applicable(work, ["업체명", "식품중분류명"])
clean = work[fd.OUTPUT_COLUMNS].reset_index(drop=True)

print(f"컬럼 수: {raw.shape[1]} -> {clean.shape[1]}")
print(f"영양성분 컬럼: 원본 {len(fd.NUTRITION_COLUMNS_ALL)}개 -> 유지 {len(fd.NUTRITION_COLUMNS_SELECTED)}개")
clean.head(3).T

컬럼 수: 50 -> 27
영양성분 컬럼: 원본 24개 -> 유지 9개


,0,1,2
식품코드,D504-212000000-0001,D104-198000000-0001,D404-198000000-0001
식품명,흰죽,흑임자죽,흑임자 죽
메뉴명,흰죽,흑임자죽,흑임자 죽
이름접두어,NaN,NaN,NaN
온도,NaN,NaN,NaN
사이즈,NaN,NaN,NaN
메뉴그룹,식사,식사,식사
식품대분류명,죽 및 스프류,죽 및 스프류,죽 및 스프류
대표식품명,흰죽,흑임자죽,흑임자 죽
식품중분류명,NaN,NaN,NaN


In [28]:
# 영양성분 결측은 유지되었는지 확인 (0으로 채우지 않음)
pd.DataFrame({
    "결측 수": clean[fd.NUTRITION_COLUMNS_SELECTED].isna().sum(),
    "결측 비율(%)": (clean[fd.NUTRITION_COLUMNS_SELECTED].isna().mean() * 100).round(1),
    "0인 행 수": (clean[fd.NUTRITION_COLUMNS_SELECTED] == 0).sum(),
})

,결측 수,결측 비율(%),0인 행 수
에너지(kcal),0,0.0,160
단백질(g),0,0.0,672
지방(g),13319,70.1,137
탄수화물(g),12777,67.3,44
당류(g),149,0.8,755
식이섬유(g),15278,80.4,108
나트륨(mg),61,0.3,581
콜레스테롤(mg),13486,71.0,1256
포화지방산(g),592,3.1,1460


## 6. 저장, 무결성 검증, 전후 비교

전체 파이프라인을 `fd.preprocess`로 한 번에 실행해 위 단계별 결과와 동일한지 확인한 뒤 저장한다.

- `food_clean.csv`: 전체 그룹 정제 결과
- `food_menu.csv`: 1차 추천 대상 (메뉴그룹 = 식사)
- `dedup_map.csv`: 통합된 행의 원본 식품코드와 대표 식품코드

In [29]:
clean_pipeline, dedup_map_pipeline = fd.preprocess(raw)

pd.testing.assert_frame_equal(clean, clean_pipeline)
pd.testing.assert_frame_equal(dedup_map, dedup_map_pipeline)
print("단계별 결과와 파이프라인 결과 일치")

단계별 결과와 파이프라인 결과 일치


### 무결성 검증

- 원본: 파이프라인 실행 후 메모리의 `raw`가 디스크의 CSV와 같은지 확인
- 중복 매핑: 유지된 식품코드와 제거된 식품코드가 겹치지 않고, 합치면 원본 전체와 같으며, 대표식품코드는 모두 유지된 행에 있어야 한다

In [30]:
pd.testing.assert_frame_equal(raw, fd.load_raw(RAW_PATH))
print("원본 CSV 변경 없음")

kept_codes = set(clean["식품코드"])
dropped_codes = set(dedup_map["식품코드"])
assert kept_codes.isdisjoint(dropped_codes), "유지 행과 제거 행이 겹침"
assert kept_codes | dropped_codes == set(raw["식품코드"]), "원본 식품코드와 불일치"
assert set(dedup_map["대표식품코드"]) <= kept_codes, "대표식품코드가 유지 행에 없음"
assert clean["식품코드"].is_unique and dedup_map["식품코드"].is_unique
print(f"중복 매핑 무결성 확인: 유지 {len(kept_codes):,} + 제거 {len(dropped_codes):,} = 원본 {len(raw):,}")

원본 CSV 변경 없음
중복 매핑 무결성 확인: 유지 18,998 + 제거 619 = 원본 19,617


In [31]:
menu = fd.select_meal_menu(clean)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(PROCESSED_DIR / "food_clean.csv", index=False, encoding="utf-8-sig")
menu.to_csv(PROCESSED_DIR / "food_menu.csv", index=False, encoding="utf-8-sig")
dedup_map.to_csv(PROCESSED_DIR / "dedup_map.csv", index=False, encoding="utf-8-sig")

for f in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"{f.name}: {f.stat().st_size / 1024:.0f} KB")

dedup_map.csv: 73 KB
food_clean.csv: 5134 KB
food_menu.csv: 2150 KB


### 행 수 비교

In [32]:
pd.DataFrame({
    "행 수": {
        "원본": len(raw),
        "중복 통합 후 (food_clean)": len(clean),
        "1차 추천 대상 (food_menu)": len(menu),
        "통합 매핑 (dedup_map)": len(dedup_map),
    }
})

,행 수
원본,19617
중복 통합 후 (food_clean),18998
1차 추천 대상 (food_menu),8032
통합 매핑 (dedup_map),619


In [33]:
clean["메뉴그룹"].value_counts()

메뉴그룹
식사     8032
음료     5768
디저트    3832
반찬     1350
기타       16
Name: count, dtype: int64

### 결측치 비교

In [34]:
shared_cols = [c for c in clean.columns if c in raw.columns]
pd.DataFrame({
    "원본 결측(%)": (raw[shared_cols].isna().mean() * 100).round(1),
    "정제 후 결측(%)": (clean[shared_cols].isna().mean() * 100).round(1),
    "식사 대상 결측(%)": (menu[shared_cols].isna().mean() * 100).round(1),
})

,원본 결측(%),정제 후 결측(%),식사 대상 결측(%)
식품코드,0.0,0.0,0.0
식품명,0.0,0.0,0.0
식품대분류명,0.0,0.0,0.0
대표식품명,0.0,0.0,0.0
식품중분류명,0.0,85.9,80.2
식품기원명,0.0,0.0,0.0
업체명,0.0,18.6,26.3
출처명,0.0,0.0,0.0
데이터생성방법명,0.0,0.0,0.0
영양성분함량기준량,0.0,0.0,0.0


In [35]:
# 새로 추가된 컬럼의 결측 (온도, 사이즈, 접두어는 해당 없으면 결측이 정상)
new_cols = ["메뉴명", "이름접두어", "온도", "사이즈", "중량값", "중량단위", "메뉴그룹", "프랜차이즈여부"]
(clean[new_cols].isna().mean() * 100).round(1).rename("정제 후 결측(%)")

메뉴명         0.0
이름접두어      18.6
온도         82.4
사이즈        64.1
중량값         0.3
중량단위        0.3
메뉴그룹        0.0
프랜차이즈여부     0.0
Name: 정제 후 결측(%), dtype: float64

### 중복 비교

In [36]:
pd.DataFrame({
    "원본": {
        "완전 중복": raw.duplicated().sum(),
        "식품명 중복": raw.duplicated("식품명").sum(),
        "식품명 + 업체명 중복": raw.duplicated(["식품명", "업체명"]).sum(),
    },
    "정제 후": {
        "완전 중복": clean.duplicated().sum(),
        "식품명 중복": clean.duplicated("식품명").sum(),
        "식품명 + 업체명 중복": clean.duplicated(["식품명", "업체명"]).sum(),
    },
})

,원본,정제 후
완전 중복,0,0
식품명 중복,3970,3351
식품명 + 업체명 중복,2236,1617


정제 후에도 식품명 중복이 남아 있는 것은 의도한 결과다. 같은 이름이라도 업체나 영양성분이 다르면 별개 행으로 둔다.

남은 `식품명 + 업체명` 중복이 실제로 무엇인지 확인한다. 사이즈나 온도가 다른 프랜차이즈 변형은 `식품명` 자체가 다르므로 이 집계에는 포함되지 않는다.

In [37]:
dup_name_company = clean[clean.duplicated(["식품명", "업체명"], keep=False)]
print(f"남은 식품명 + 업체명 중복 행: {len(dup_name_company):,}")
dup_name_company["프랜차이즈여부"].value_counts().rename("행 수")

남은 식품명 + 업체명 중복 행: 2,476


프랜차이즈여부
False    2455
True       21
Name: 행 수, dtype: int64

대부분은 비프랜차이즈 데이터로, 같은 음식이 급식 종류나 외식 등 출처별로 다른 중량과 영양성분으로 등록된 경우다. 영양성분이 다르므로 통합하지 않았다.

In [38]:
dup_name_company.loc[~dup_name_company["프랜차이즈여부"], ["식품명", "식품기원명", "중량값", "중량단위", "에너지(kcal)", "나트륨(mg)"]].head(8)

,식품명,식품기원명,중량값,중량단위,에너지(kcal),나트륨(mg)
3,흑미밥_찹쌀,초등학교급식(재료량 기반 산출 함량),190.0,ml,118,2.0
4,흑미밥_찹쌀,중고등학교급식(재료량 기반 산출함량),270.0,ml,121,2.0
5,흑미밥,산업체급식(재료량 기반 산출 함량),390.0,ml,118,3.0
6,흑미밥,중고등학교급식(재료량 기반 산출함량),310.0,ml,118,3.0
7,흑미밥,초등학교급식(재료량 기반 산출 함량),250.0,ml,120,1.0
8,흑미밥,외식(재료량 기반 산출함량),300.0,ml,120,1.0
10,훈제오리,외식(분석함량),NaN,NaN,259,888.0
11,훈제오리,외식(분석함량),250.0,g,319,488.0


프랜차이즈에서 남은 소수는 같은 브랜드의 간편조리세트 메뉴가 중량이 다른 여러 행으로 등록된 경우다.

In [39]:
dup_name_company.loc[dup_name_company["프랜차이즈여부"], ["식품명", "업체명", "중량값", "중량단위", "에너지(kcal)"]].head(6)

,식품명,업체명,중량값,중량단위,에너지(kcal)
503,햄버거_간편조리세트_햄버거,맥도날드,244.0,g,223
505,햄버거_간편조리세트_햄버거,버거킹,409.0,g,241
506,햄버거_간편조리세트_햄버거,맥도날드,242.0,g,230
507,햄버거_간편조리세트_햄버거,맥도날드,213.0,g,256
508,햄버거_간편조리세트_햄버거,버거킹,293.0,g,236
509,햄버거_간편조리세트_치킨버거,롯데리아,141.0,g,243


### 대표 샘플 비교

In [40]:
compare_cols = ["식품코드", "식품명", "메뉴명", "이름접두어", "온도", "사이즈", "메뉴그룹", "업체명", "중량값", "중량단위"]
sample_codes = ["D504-212000000-0001", raw.loc[raw["식품명"].str.contains("아메리카노 아이스"), "식품코드"].iloc[0], raw.loc[raw["식품명"] == "된장국_근대", "식품코드"].iloc[0]]

print("원본")
display(raw.loc[raw["식품코드"].isin(sample_codes), ["식품코드", "식품명", "식품대분류명", "업체명", "식품중량"]])
print("정제 후")
display(clean.loc[clean["식품코드"].isin(sample_codes), compare_cols])

원본


,식품코드,식품명,식품대분류명,업체명,식품중량
0,D504-212000000-0001,흰죽,죽 및 스프류,해당없음,291.90ml
7079,D220-748080000-0117,커피_화이트 아메리카노 아이스(ICED) (Venti),음료 및 차류,커피에반하다,720ml
17208,D605-216050000-0001,된장국_근대,국 및 탕류,해당없음,200ml


정제 후


,식품코드,식품명,메뉴명,이름접두어,온도,사이즈,메뉴그룹,업체명,중량값,중량단위
0,D504-212000000-0001,흰죽,흰죽,NaN,NaN,NaN,식사,NaN,291.9,ml
7001,D220-748080000-0117,커피_화이트 아메리카노 아이스(ICED) (Venti),화이트 아메리카노,커피,ICED,Venti,음료,커피에반하다,720.0,ml
16759,D605-216050000-0001,된장국_근대,된장국 근대,NaN,NaN,NaN,식사,NaN,200.0,ml


In [41]:
menu.sample(10, random_state=3)[["식품코드", "메뉴명", "식품대분류명", "프랜차이즈여부", "업체명", "에너지(kcal)", "나트륨(mg)"]]

,식품코드,메뉴명,식품대분류명,프랜차이즈여부,업체명,에너지(kcal),나트륨(mg)
1089,D202-120000000-3591,페퍼로니 피자 치즈버스트 나폴리,빵 및 과자류,True,도미노피자,284,557.0
202,D407-349000000-0001,해물콩나물찜,찜류,False,NaN,51,139.0
1729,D202-120340000-0107,치즈피자 오리지널,빵 및 과자류,True,비스트로피자,302,403.0
7091,D708-375020000-0001,돼지불고기 고추장,구이류,False,NaN,85,218.0
4010,D202-120000000-0478,리얼불고기 피자 트리플치즈버스트 엣지,빵 및 과자류,True,도미노피자,248,459.0
2379,D202-120000000-4542,야채,빵 및 과자류,True,봉수아피자,164,203.0
5460,D603-164000000-0001,우동,면 및 만두류,False,NaN,61,169.0
6477,D202-091000000-0286,청양칠리 새우 베이컨,빵 및 과자류,True,롯데리아,222,502.0
2724,D202-120000000-1011,스윗고구마 피자,빵 및 과자류,True,뽕뜨락피자,255,306.0
2753,D202-120000000-4435,슈프림,빵 및 과자류,True,봉수아피자,217,344.0


## 7. 식사 후보 분포와 라벨링 샘플 방안

1차 추천 대상 `food_menu.csv`의 분포를 확인한다. 특히 프랜차이즈 피자가 얼마나 편중되어 있는지 본다. 데이터를 삭제하지 않고 분포만 확인한다.

In [42]:
pd.crosstab(menu["식품대분류명"], menu["프랜차이즈여부"], margins=True).sort_values("All", ascending=False)

프랜차이즈여부,False,True,All
식품대분류명,,,
All,2111,5921,8032
빵 및 과자류,33,5429,5462
국 및 탕류,419,10,429
밥류,346,26,372
튀김류,109,257,366
면 및 만두류,243,70,313
찌개 및 전골류,271,39,310
볶음류,180,47,227
구이류,191,24,215


In [43]:
menu["대표식품명"].value_counts().head(15)

대표식품명
피자        4710
버거         295
닭튀김        251
샌드위치       233
햄버거        129
핫도그         61
스파게티        54
볶음밥         53
떡볶이         50
된장국         43
토스트         34
김밥          33
김치찌개        29
된장찌개        28
돼지고기볶음      28
Name: count, dtype: int64

In [44]:
pizza = menu[menu["대표식품명"] == "피자"]
print(f"피자 행: {len(pizza):,} / 식사 {len(menu):,} ({len(pizza) / len(menu) * 100:.1f}%)")
print(f"피자 업체 수: {pizza['업체명'].nunique()}")
print(f"피자 메뉴명 고유값 (사이즈 제거 후): {pizza['메뉴명'].nunique():,}")
print()
print(pizza["사이즈"].value_counts(dropna=False))

피자 행: 4,710 / 식사 8,032 (58.6%)
피자 업체 수: 55
피자 메뉴명 고유값 (사이즈 제거 후): 2,614

사이즈
L      2177
R       962
NaN     574
M       485
F       227
P       153
XL      103
G        28
J         1
Name: count, dtype: int64


피자는 사이즈를 분리해도 메뉴명이 2,600개가 넘는다. 도우, 엣지, 토핑 조합이 이름에 들어가기 때문이다.

### 라벨링 단위

라벨은 음식 자체에 붙는 정보지만, 같은 메뉴명이나 대표식품명이라는 이유만으로 공유하면 업체나 문맥이 다른 메뉴가 같은 라벨을 받는다. 예를 들어 `아메리카노`는 업체마다 다른 상품이고, `피자`라는 대표식품명 아래에는 매운 피자와 순한 피자가 함께 있다.

따라서 라벨링 단위는 다음 여섯 값이 모두 같은 행의 묶음으로 정한다. 사이즈나 출처(급식 종류)만 다른 행이 하나의 단위가 된다.

- `메뉴명`, `이름접두어`, `대표식품명`, `식품대분류명`, `업체명`, `온도`

이 기준은 `src/labeling/sampling.py`의 `UNIT_KEY_COLUMNS`와 같다. 단위 수를 확인한다.

In [45]:
UNIT_KEY = ["메뉴명", "이름접두어", "대표식품명", "식품대분류명", "업체명", "온도"]
units = menu.drop_duplicates(UNIT_KEY)
print(f"식사 행: {len(menu):,} -> 라벨링 단위: {len(units):,}")
print(f"비프랜차이즈 단위: {(~units['프랜차이즈여부']).sum():,}, 프랜차이즈 단위: {units['프랜차이즈여부'].sum():,}")
units["대표식품명"].value_counts().head(10)

식사 행: 8,032 -> 라벨링 단위: 5,219
비프랜차이즈 단위: 1,123, 프랜차이즈 단위: 4,096


대표식품명
피자      2915
버거       283
닭튀김      232
샌드위치     231
햄버거      119
핫도그       61
스파게티      50
떡볶이       44
토스트       34
볶음밥       28
Name: count, dtype: int64

### 라벨링 진행 원칙

- 라벨은 위 단위 안에서만 공유하고, 대표식품명이나 메뉴명이 같다는 이유로 다른 단위에 복사하지 않는다.
- 라벨링하지 않은 단위는 미라벨링 상태로 남긴다. 대표식품의 라벨을 자동으로 채우지 않는다.
- 각 단위는 포함된 원본 식품코드 목록을 가지므로 결과를 행 단위로 되돌릴 수 있다.
- 실험 단계에서는 약 100개 단위를 식품대분류별로 배분하고 대표식품명당 상한을 두어 피자 편중을 줄인다. 실험 샘플 추출과 프롬프트, 검증은 `03_llm_labeling.ipynb`에서 진행한다.
- 전체 5천여 단위는 실험 결과를 검토한 뒤 Batch API로 확장한다.

## 정리

### 결정한 기준과 근거

- 1차 추천 대상은 메뉴그룹 `식사`다. 빵 및 과자류 중 피자, 버거, 햄버거, 샌드위치, 핫도그, 토스트는 한 끼 식사로 먹는 메뉴라 식사에 포함했다. 반찬은 단독 추천 메뉴로 부적절해 제외했다. 디저트, 음료, 기타는 라벨만 남겼다.
- 볶음류, 조림류, 구이류, 튀김류, 찜류, 전·적 및 부침류는 대표식품명 키워드로 반찬을 다시 골라냈다. 강한 반찬 키워드, 식사 키워드, 반찬 키워드 순으로 판단해 `감자그라탕`, `마파두부`처럼 재료명만 겹치는 식사가 빠지지 않게 했다. 프랜차이즈 사이드 메뉴도 반찬 그룹에 넣었다.
- 식품명은 원본을 보존하고 `메뉴명`을 추가했다. 프랜차이즈 접두어는 `이름접두어`, HOT/ICED는 `온도`, 사이즈 표기는 `사이즈`로 분리했다. 수량 표기(조각, N개입)는 메뉴 정보라 이름에 남겼다.
- 중복은 식품명, 업체명, 기준량, 중량, 영양성분 24개가 모두 같을 때만 통합했다. 결과적으로 급식 반복 데이터만 통합되었고 프랜차이즈 메뉴는 하나도 제거되지 않았다. 통합된 행은 `dedup_map.csv`로 추적한다.
- 영양성분 결측은 채우지 않았다. `해당없음`은 업체명과 중분류명에서 결측으로 바꾸고 소분류·세분류는 컬럼을 제외했다.
- 식품중량은 숫자와 단위로만 분리했다. 1인분 여부와 g/ml 환산은 근거가 없어 하지 않았다.

- 남은 식품명 중복은 대부분 같은 음식이 출처별로 다른 영양성분으로 등록된 비프랜차이즈 행이다. 영양성분이 다르므로 통합하지 않았다.
- 피자 편중은 데이터 삭제 대신 라벨링 단위(메뉴명, 접두어, 대표식품명, 대분류, 업체, 온도가 모두 같은 묶음)와 대표식품명별 샘플 상한으로 대응한다. 라벨은 단위 밖으로 공유하지 않는다.

### 남은 판단 사항

- 반찬 키워드 규칙은 현재 대표식품명 441개를 기준으로 정한 것이라 새 데이터가 들어오면 다시 검토해야 한다. 전류(동태전, 굴전 등)와 안주 성격 메뉴(닭발, 곱창)는 경계가 애매해 현재 판단을 그대로 둔다.
- 프랜차이즈 피자가 식사 그룹의 절반 이상이다. 추천 시 `프랜차이즈여부` 가중치 또는 제외 여부는 추천 단계에서 확정한다.
- 실험 라벨링 결과를 검토한 뒤 전체 단위를 Batch API로 라벨링할지, 프랜차이즈 단위를 어느 범위까지 포함할지 결정해야 한다.
- 사이즈만 다른 프랜차이즈 메뉴(예: 치즈 피자 L, M, R)를 하나의 메뉴로 볼지, 별개 후보로 둘지 추천 단계에서 결정해야 한다.
- 비프랜차이즈 메뉴명의 재료 변형(`된장국 근대`)을 임베딩 텍스트에 그대로 쓸지, 기본 메뉴명과 재료를 분리할지는 임베딩 실험에서 확인한다.
- `영양성분함량기준량`이 100g 또는 100ml 기준이라 메뉴 간 영양 비교는 중량을 곱한 총량 기준이 필요할 수 있다. 다만 `식품중량`이 1인 제공량인지 확인한 뒤 진행해야 한다.
- 프랜차이즈 데이터의 미량영양소 결측은 측정하지 않은 값이므로 영양 기반 필터에서는 결측을 조건 미충족이 아닌 미확인으로 다뤄야 한다.
- 매운맛, 국물, 온도, 조리법 등 자연어 추천에 필요한 속성은 다음 LLM 라벨링 단계에서 `food_menu.csv`를 기준으로 생성한다.